In [ ]:
!pip install numpy scipy matplotlib cupy-cuda12x --break-system-packages
!pip install tqdm

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.constants import c as c_SI, epsilon_0, m_e, elementary_charge as q_e
from scipy.interpolate import interp1d
import sys, json
from tqdm.auto import tqdm
sys.path.insert(0, '/workspace/files')
print('Imports OK')

Imports OK


In [ ]:
from pathlib import Path
import numpy as np
from scipy.constants import c as c_SI, epsilon_0, m_e
from scipy.constants import elementary_charge as q_e

from NewSim3juillet import run

OUT_DIR  = '/workspace/sim_filament_13uJ_w6'
NPZ_PATH = Path(OUT_DIR) / 'result.npz'
EXP_DIR  = '/workspace/data_exp'

PAS_FS       = 67.0
DELAYS_STEPS = np.arange(-20, 20)
DELAYS_FS    = DELAYS_STEPS * PAS_FS

Z_FOCUS_GLASS_DIST_UM = 394.0
BEGIN_M               = -Z_FOCUS_GLASS_DIST_UM * 1e-6
END_M                 =  800e-6

N0_PUMP         = 1.4500
ENERGY_INPUT_UJ = 13.0
TRANSMISSION    = 1.0 - ((N0_PUMP - 1.0) / (N0_PUMP + 1.0))**2
ENERGY_IN_GLASS = ENERGY_INPUT_UJ * TRANSMISSION

GAUSS_FLATTOP = float(np.sqrt(np.pi / (4.0 * np.log(2))))
ENERGY_SIM_UJ = ENERGY_IN_GLASS / GAUSS_FLATTOP

W0_M = 3e-6
DELTA_T_S = 263e-15

LZ = (END_M - BEGIN_M)
NZ = int(LZ / 24e-9)

SHEAR_UM   = 2.0
Z_RANGE_UM = (-Z_FOCUS_GLASS_DIST_UM, 500.0)
X_LIM_UM   = (-50, 50)

LAMBDA_PROBE = 490e-9
N0_PROBE     = 1.4629
NC_PROBE_CM3 = epsilon_0 * m_e * (2*np.pi*c_SI/LAMBDA_PROBE)**2 / q_e**2 * 1e-6
c_um_fs      = c_SI * 1e-9

def _fmt(t):
    if t >= 1e12: return f'{t*1e-12:.1f} ms'
    if t >= 1e9:  return f'{t*1e-9:.1f} µs'
    if t >= 1e6:  return f'{t*1e-6:.1f} ns'
    if t >= 1e3:  return f'{t*1e-3:.1f} ps'
    return f'{t:.0f} fs'

print(f'Delays:    {DELAYS_FS[0]:.0f} to {DELAYS_FS[-1]:.0f} fs')
print(f'Probe:     {LAMBDA_PROBE*1e9:.0f} nm, nc={NC_PROBE_CM3:.3e} cm^-3')
print(f'Geometry:  interface→focus = {Z_FOCUS_GLASS_DIST_UM:.0f} µm in glass')
print(f'           sim z range     = [{BEGIN_M*1e6:.0f}, {END_M*1e6:.0f} µm')
print(f'Pulse:     {ENERGY_IN_GLASS:.2f} µJ dans la boîte '
      f'(energy_uJ={ENERGY_SIM_UJ:.2f}, T={TRANSMISSION*100:.2f}%), w0={W0_M*1e6:.1f} µm, '
      f'FWHM={DELTA_T_S*1e15:.0f} fs')
print(f'NZ:        {NZ}  ({LZ*1e9/NZ:.0f} nm/step)')
print(f'NPZ exists:{NPZ_PATH.exists()}')

In [ ]:
if NPZ_PATH.exists():
    print(f'Loading existing results: {NPZ_PATH}')
    res = dict(np.load(NPZ_PATH, allow_pickle=True))
else:
    print("Launching simulation...")
    res = run(
        # Grid
        Nz=NZ, Nt=2000, Nr=3001,        # Nr = ordre Hankel N (3000 pts radiaux)
        begin=BEGIN_M, end=END_M,
        R_factor=90.0,                  # boîte 270 µm, absorbeur à 243 µm
        # Laser
        wavelength=1030e-9,
        energy_uJ=ENERGY_SIM_UJ,        # 11.80 -> 12.56 µJ réellement lancés
        w0=W0_M,
        delta_t=DELTA_T_S,
        # Material
        rho_max=2.1e22,
        # Physics
        enable_ste=True,
        tau_r=330e-15,                  # Mouskeftaras 2013 / Tsaturyan 2025
        # Probe
        lambda_probe=490e-9,
        rho_t_stride=10,                # 200 snapshots, dt_sub = 35.6 fs
        # Recording
        save_stride=100,                # 497 plans z, dz_save = 2.4 µm
        out_dir=OUT_DIR,
        envelope='gaussian_focused',
    )
    print(f'Done -> {OUT_DIR}/')

In [ ]:
z_um     = np.asarray(res['z'])     * 1e6
# Gestion robuste du nom de variable radial (rlist ou r)
if 'rlist' in res:
    rlist_um = np.asarray(res['rlist']) * 1e6
elif 'r' in res:
    rlist_um = np.asarray(res['r']) * 1e6
else:
    rlist_um = None

t_sub_fs = np.asarray(res['t_sub_fs']) if 't_sub_fs' in res else None
Imax_z   = np.asarray(res['Imax_z'])

t_sim_min, t_sim_max = (float(t_sub_fs[0]), float(t_sub_fs[-1])) if t_sub_fs is not None else (0,0)

print("--- Résumé des données ---")
if z_um is not None:
    print(f'z     : [{z_um[0]:.0f}, {z_um[-1]:.0f}] µm  ({len(z_um)} planes)')
if rlist_um is not None:
    print(f'r     : [{rlist_um[0]:.1f}, {rlist_um[-1]:.1f}] µm')
if t_sub_fs is not None:
    print(f't_sub : [{t_sim_min:.0f}, {t_sim_max:.0f} fs  ({len(t_sub_fs)} snapshots)')
    
print(f'I_max : {Imax_z.max():.3e} W/cm²  @ z={z_um[Imax_z.argmax()]:.0f} µm')

if "I_rzt" in res and res["I_rzt"] is not None:
    print("I_rzt : présent")
else:
    print("I_rzt : absent (rerun for Kerr)")